# Bronze — Ingesta Kafka → Delta Lake
**LogiLake | D'LOGIA** — Capa Bronze del Medallion Architecture

Este notebook consume mensajes del topic `olist_orders` en Kafka usando
**Spark Structured Streaming** y los persiste en Delta Lake como datos raw.

Prerrequisitos:
- `pip install -r requirements.txt`
- Kafka corriendo local: `cd kafka && docker-compose up -d`
- Topic creado: `bash kafka/topic_config.sh create`
- Producer corriendo: `python kafka/olist_producer.py --data_path data/raw`

Para ingesta sin Kafka, saltar a la celda **Alternativa batch**.

In [1]:
# ── SparkSession con Delta Lake 3.1.0 + Kafka (PySpark 3.5.0) ────────────────
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("logilake_bronze")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true")
    # Reduce verbosidad en local
    .config("spark.ui.showConsoleProgress", "false")
)

# Incluye Kafka connector para Structured Streaming
spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"]
).getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | Delta Lake activo")

Spark 3.5.0 | Delta Lake activo


In [2]:
# ── Configuracion de rutas locales ────────────────────────────────────────────
import os

BASE_PATH       = "./data"
BRONZE_PATH     = f"{BASE_PATH}/bronze/olist_orders"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze_olist"
RAW_DATA_PATH   = f"{BASE_PATH}/raw"

# Bootstrap server de Kafka
# Opcion A - Docker en misma maquina: 'localhost:9092'
# Opcion B - Docker con host.docker.internal: 'host.docker.internal:29092'
KAFKA_BOOTSTRAP = "localhost:9092"
KAFKA_TOPIC     = "olist_orders"

os.makedirs(BRONZE_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

print(f"Bronze path:     {BRONZE_PATH}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(f"Kafka broker:    {KAFKA_BOOTSTRAP}")

Bronze path:     ./data/bronze/olist_orders
Checkpoint path: ./data/checkpoints/bronze_olist
Kafka broker:    localhost:9092


In [4]:
# ── Schema del payload Kafka ──────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType, ArrayType
)

OLIST_SCHEMA = StructType([
    StructField("event_id",             StringType(),  True),
    StructField("order_id",             StringType(),  False),
    StructField("customer_id",          StringType(),  True),
    StructField("order_status",         StringType(),  True),
    StructField("order_purchase_ts",    StringType(),  True),
    StructField("order_approved_ts",    StringType(),  True),
    StructField("order_delivered_ts",   StringType(),  True),
    StructField("order_estimated_ts",   StringType(),  True),
    StructField("item_count",           IntegerType(), True),
    StructField("categories",           ArrayType(StringType()), True),
    StructField("seller_states",        ArrayType(StringType()), True),
    StructField("total_items_value",    FloatType(),   True),
    StructField("total_freight",        FloatType(),   True),
    StructField("payment_type",         StringType(),  True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value",        FloatType(),   True),
    StructField("review_score",         FloatType(),   True),
    StructField("ingested_at",          StringType(),  True),
    StructField("source",               StringType(),  True),
])

print("Schema definido:", len(OLIST_SCHEMA.fields), "campos")

Schema definido: 19 campos


In [7]:
# ── Leer stream desde Kafka ───────────────────────────────────────────────────
# Ejecutar solo si Kafka esta corriendo.
# Si no tienes Kafka, saltar a la celda de ingesta batch.

kafka_stream_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", 10_000)
    .option("failOnDataLoss", "false")
    .load()
)

# Deserializar JSON + metadatos Kafka
bronze_stream = (
    kafka_stream_raw
    .withColumn("value_str", F.col("value").cast("string"))
    .withColumn("data", F.from_json(F.col("value_str"), OLIST_SCHEMA))
    .select(
        "data.*",
        F.col("topic").alias("kafka_topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.current_timestamp().alias("bronze_ingested_at"),
    )
    .filter(F.col("order_id").isNotNull())
)

# Escribir a Delta Lake Bronze (streaming)
bronze_query = (
    bronze_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(processingTime="30 seconds")
    .start(BRONZE_PATH)
)

print(f"Stream iniciado. ID: {bronze_query.id}")
# bronze_query.awaitTermination()  # Descomentar para run continuo

Stream iniciado. ID: dc7a948f-54a3-4e10-8aba-d34c506c0d05


In [6]:
# ── Monitoreo del stream ──────────────────────────────────────────────────────
import time
time.sleep(35)

print("Estado:", bronze_query.status)
if bronze_query.recentProgress:
    last = bronze_query.recentProgress[-1]
    print(f"Rows procesados: {last.get('numInputRows', 0)}")
    print(f"Throughput:      {last.get('processedRowsPerSecond', 0):.1f} rows/s")
    print(f"Batch ID:        {last.get('batchId', 'N/A')}")

# bronze_query.stop()

Estado: {'message': "Terminated with exception: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'", 'isDataAvailable': False, 'isTriggerActive': False}


In [ ]:
# ── Alternativa: Ingesta batch desde CSV (sin Kafka) ─────────────────────────
# Usar este bloque si Kafka no esta disponible.
# Copia los CSVs de Olist a: data/raw/

def load_olist_batch(raw_path: str):
    orders   = spark.read.csv(f"{raw_path}/olist_orders_dataset.csv",                  header=True, inferSchema=True)
    items    = spark.read.csv(f"{raw_path}/olist_order_items_dataset.csv",              header=True, inferSchema=True)
    payments = spark.read.csv(f"{raw_path}/olist_order_payments_dataset.csv",           header=True, inferSchema=True)
    reviews  = spark.read.csv(f"{raw_path}/olist_order_reviews_dataset.csv",            header=True, inferSchema=True)
    products = spark.read.csv(f"{raw_path}/olist_products_dataset.csv",                 header=True, inferSchema=True)
    category = spark.read.csv(f"{raw_path}/product_category_name_translation.csv",      header=True, inferSchema=True)
    sellers  = spark.read.csv(f"{raw_path}/olist_sellers_dataset.csv",                  header=True, inferSchema=True)

    items_rich = (
        items
        .join(products.join(category, "product_category_name", "left"), "product_id", "left")
        .join(sellers.select("seller_id", "seller_state", "seller_city"), "seller_id", "left")
    )

    items_agg = items_rich.groupBy("order_id").agg(
        F.count("order_item_id").alias("item_count"),
        F.sum("price").alias("total_items_value"),
        F.sum("freight_value").alias("total_freight"),
        F.collect_set("product_category_name_english").alias("categories"),
        F.collect_set("seller_state").alias("seller_states"),
    )

    payments_agg = payments.groupBy("order_id").agg(
        F.first("payment_type").alias("payment_type"),
        F.max("payment_installments").alias("payment_installments"),
        F.sum("payment_value").alias("payment_value"),
    )

    reviews_latest = (
        reviews
        .orderBy("review_creation_date")
        .dropDuplicates(["order_id"])
        .select("order_id", "review_score")
    )

    return (
        orders
        .join(items_agg, "order_id", "left")
        .join(payments_agg, "order_id", "left")
        .join(reviews_latest, "order_id", "left")
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("source", F.lit("batch_csv_v1"))
    )


bronze_batch_df = load_olist_batch(RAW_DATA_PATH)
bronze_batch_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(BRONZE_PATH)

print(f"Bronze batch OK: {bronze_batch_df.count():,} filas → {BRONZE_PATH}")

In [ ]:
# ── Verificacion: leer Bronze ─────────────────────────────────────────────────
bronze_df = spark.read.format("delta").load(BRONZE_PATH)

print(f"Total registros en Bronze: {bronze_df.count():,}")
bronze_df.select(
    "order_id", "order_status", "order_purchase_ts",
    "item_count", "payment_value", "review_score",
    "bronze_ingested_at"
).show(10, truncate=False)

print("\nDistribucion de estados:")
bronze_df.groupBy("order_status").count().orderBy(F.desc("count")).show()